In [2]:
!git clone https://github.com/swz30/MPRNet.git
!git clone https://github.com/xinntao/ESRGAN.git
!git clone https://github.com/xinntao/BasicSR.git

Cloning into 'MPRNet'...
remote: Enumerating objects: 339, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 339 (delta 65), reused 56 (delta 56), pack-reused 259 (from 1)
Receiving objects: 100% (339/339), 92.17 KiB | 1.96 MiB/s, done.
Resolving deltas: 100% (187/187), done.
Cloning into 'ESRGAN'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 225 (delta 15), reused 14 (delta 14), pack-reused 205 (from 1)
Receiving objects: 100% (225/225), 24.86 MiB | 45.21 MiB/s, done.
Resolving deltas: 100% (85/85), done.
Cloning into 'BasicSR'...
remote: Enumerating objects: 5924, done.
remote: Total 5924 (delta 0), reused 0 (delta 0), pack-reused 5924 (from 1)
Receiving objects: 100% (5924/5924), 4.14 MiB | 19.80 MiB/s, done.
Resolving deltas: 100% (3757/3757), done.


In [3]:
!gdown --id 1LODPt9kYmxwU98g96UrRA0_Eh5HYcsRw -O /kaggle/working/MPRNet/Denoising/pretrained_models/MPRNet_Denoising.pth
!gdown --id 1pJ_T-V1dpb1ewoEra1TGSWl5e6H7M4NN -O /kaggle/working/ESRGAN/models/RRDB_ESRGAN_x4.pth

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1LODPt9kYmxwU98g96UrRA0_Eh5HYcsRw
To: /kaggle/working/MPRNet/Denoising/pretrained_models/MPRNet_Denoising.pth
100%|███████████████████████████████████████| 63.0M/63.0M [00:00<00:00, 103MB/s]
/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1pJ_T-V1dpb1ewoEra1TGSWl5e6H7M4NN
To: /kaggle/working/ESRGAN/models/RRDB_ESRGAN_x4.pth
100%|███████████████████████████████████████| 66.9M/66.9M [00:00<00:00, 257MB/s]


In [4]:
import os, glob, math
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import cv2
import torch
import os, glob, math, random
from pathlib import Path
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import sys

sys.path.append('/kaggle/working/ESRGAN')
from RRDBNet_arch import RRDBNet
from MPRNet.Denoising.MPRNet import MPRNet as MPRNetClass


In [5]:
ROOT = '/kaggle/input/dlp-may-2025-nppe-3/archive'
TRAIN_LR_DIR = "/kaggle/input/dlp-may-2025-nppe-3/archive/train/train"
TRAIN_HR_DIR = "/kaggle/input/dlp-may-2025-nppe-3/archive/train/gt"
VAL_LR_DIR   = "/kaggle/input/dlp-may-2025-nppe-3/archive/val/val"
VAL_HR_DIR   = "/kaggle/input/dlp-may-2025-nppe-3/archive/val/gt"
TEST_DIR     = "/kaggle/input/dlp-may-2025-nppe-3/archive/test"

OUT_ROOT = 'outputs'
VAL_OUT  = 'outputs/val_out'
TEST_OUT = 'outputs/test_out'
os.makedirs(VAL_OUT, exist_ok=True)
os.makedirs(TEST_OUT, exist_ok=True)

MPRNET_PRETRAIN = '/kaggle/working/MPRNet/Denoising/pretrained_models/MPRNet_Denoising.pth'
RRDB_PRETRAIN   = '/kaggle/working/ESRGAN/models/RRDB_ESRGAN_x4.pth'

SEED = 1337
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DN_EPOCHS= 2
DN_BATCH_SIZE= 4
DN_LR= 1e-4
DN_CROP_LR= 128

SR_EPOCHS= 2
SR_BATCH_SIZE= 4
SR_LR= 1e-4
SR_CROP_LR= 64
SCALE= 4

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

In [6]:
def imread_rgb(path):
    return np.array(Image.open(path).convert('RGB'))

def to_tensor01(img_np):
    return torch.from_numpy(img_np).permute(2,0,1).float().div(255.).unsqueeze(0)

def to_image_u8(t):
    t = t.clamp(0,1).squeeze(0).permute(1,2,0).detach().cpu().numpy()
    return (t*255.0 + 0.5).astype(np.uint8)

def downscale_to_lr(hr_img_np, target_wh):
    return np.array(Image.fromarray(hr_img_np).resize(target_wh, Image.BICUBIC))

def psnr(img1_u8, img2_u8):
    mse = np.mean((img1_u8.astype(np.float32) - img2_u8.astype(np.float32)) ** 2)
    if mse == 0: return 99.0
    return 20.0 * math.log10(255.0 / math.sqrt(mse))


In [7]:
class PairedPaths:
    def __init__(self, lr_dir, hr_dir):
        self.lr_paths = sorted([p for p in glob.glob(os.path.join(lr_dir, '*')) if os.path.isfile(p)])
        self.hr_dir = hr_dir
        self.pairs = []
        for lp in self.lr_paths:
            name = os.path.basename(lp)
            hp = os.path.join(hr_dir, name)
            if not os.path.exists(hp):
                base = os.path.splitext(name)[0]
                found = None
                for ext in ['.png', '.jpg', '.jpeg', '.bmp', '.tiff']:
                    cand = os.path.join(hr_dir, base + ext)
                    if os.path.exists(cand):
                        found = cand; break
                if found is None:
                    continue
                hp = found
            self.pairs.append((lp, hp))

    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx): return self.pairs[idx]

class DenoiseDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, crop_lr=128, scale=4):
        self.pp = PairedPaths(lr_dir, hr_dir)
        self.crop = crop_lr
        self.scale = scale

    def __len__(self): return len(self.pp)

    def __getitem__(self, idx):
        lr_path, hr_path = self.pp[idx]
        lr = imread_rgb(lr_path)
        hr = imread_rgb(hr_path)

        Hlr, Wlr = lr.shape[:2]
        lr_clean = downscale_to_lr(hr, (Wlr, Hlr))

        ch = min(self.crop, Hlr)
        cw = min(self.crop, Wlr)
        y0 = 0 if Hlr==ch else np.random.randint(0, Hlr - ch + 1)
        x0 = 0 if Wlr==cw else np.random.randint(0, Wlr - cw + 1)

        lr_noisy_patch = lr[y0:y0+ch, x0:x0+cw]
        lr_clean_patch = lr_clean[y0:y0+ch, x0:x0+cw]

        lr_noisy_t = torch.from_numpy(lr_noisy_patch).permute(2,0,1).float().div(255.)
        lr_clean_t = torch.from_numpy(lr_clean_patch).permute(2,0,1).float().div(255.)
        return lr_noisy_t, lr_clean_t

class SRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, crop_lr=64, scale=4):
        self.pp = PairedPaths(lr_dir, hr_dir)
        self.crop = crop_lr
        self.scale = scale

    def __len__(self): return len(self.pp)

    def __getitem__(self, idx):
        lr_path, hr_path = self.pp[idx]
        lr = imread_rgb(lr_path)
        hr = imread_rgb(hr_path)

        Hlr, Wlr = lr.shape[:2]
        Hhr, Whr = hr.shape[:2]

        ch = min(self.crop, Hlr)
        cw = min(self.crop, Wlr)
        y0 = 0 if Hlr==ch else np.random.randint(0, Hlr - ch + 1)
        x0 = 0 if Wlr==cw else np.random.randint(0, Wlr - cw + 1)

        lr_patch = lr[y0:y0+ch, x0:x0+cw]

        ys, xs = y0*self.scale, x0*self.scale
        ch_hr, cw_hr = ch*self.scale, cw*self.scale

        if (Hhr >= ys+ch_hr) and (Whr >= xs+cw_hr):
            hr_patch = hr[ys:ys+ch_hr, xs:xs+cw_hr]
        else:
            hr_resz = np.array(Image.fromarray(hr).resize((Wlr*self.scale, Hlr*self.scale), Image.BICUBIC))
            hr_patch = hr_resz[ys:ys+ch_hr, xs:xs+cw_hr]

        lr_t = torch.from_numpy(lr_patch).permute(2,0,1).float().div(255.)
        hr_t = torch.from_numpy(hr_patch).permute(2,0,1).float().div(255.)
        return lr_t, hr_t


In [8]:
train_dn = DenoiseDataset(TRAIN_LR_DIR, TRAIN_HR_DIR, crop_lr=DN_CROP_LR, scale=SCALE)
train_sr = SRDataset(TRAIN_LR_DIR, TRAIN_HR_DIR, crop_lr=SR_CROP_LR, scale=SCALE)

dl_dn = DataLoader(train_dn, batch_size=DN_BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
dl_sr = DataLoader(train_sr, batch_size=SR_BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

len(train_dn), len(train_sr)

(1105, 1105)

In [9]:
def load_mprnet(ckpt_path):
    model = MPRNetClass()
    state = torch.load(ckpt_path, map_location='cpu')
    if isinstance(state, dict):
        for k in ['state_dict','params','model']:
            if k in state and isinstance(state[k], dict):
                state = state[k]; break
    state = {k.replace('module.',''): v for k,v in state.items()}
    model.load_state_dict(state, strict=False)
    return model

def load_rrdb_x4(ckpt_path, device):
    model = RRDBNet(in_nc=3, out_nc=3, nf=64, nb=23, gc=32)
    
    state = torch.load(ckpt_path, map_location='cpu')
    if isinstance(state, dict) and any(k in state for k in ('params_ema', 'state_dict')):
        state = state.get('params_ema', state.get('state_dict', state))
    if isinstance(state, dict):
        state = {k.replace('module.', ''): v for k, v in state.items()}

    missing, unexpected = model.load_state_dict(state, strict=False)
    model.to(device).eval()
    return model


denoiser = load_mprnet(MPRNET_PRETRAIN).to(device)
opt_dn   = torch.optim.Adam(denoiser.parameters(), lr=DN_LR)
crit_L1  = nn.L1Loss()
scaler_dn = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

sr_net  = sr_net = load_rrdb_x4("/kaggle/working/ESRGAN/models/RRDB_ESRGAN_x4.pth", device)
opt_sr  = torch.optim.Adam(sr_net.parameters(), lr=SR_LR)
scaler_sr = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

/tmp/ipykernel_19/3647206099.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_dn = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
/tmp/ipykernel_19/3647206099.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_sr = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))


In [10]:
denoiser.train()
for ep in range(1, DN_EPOCHS+1):
    pbar = tqdm(dl_dn, desc=f"MPRNet | Epoch {ep}/{DN_EPOCHS}")
    running = 0.0
    for lr_noisy_t, lr_clean_t in pbar:
        lr_noisy_t = lr_noisy_t.to(device, non_blocking=True)
        lr_clean_t = lr_clean_t.to(device, non_blocking=True)

        opt_dn.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
            out = denoiser(lr_noisy_t)
            if isinstance(out, (list, tuple)): out = out[0]
            loss = crit_L1(out, lr_clean_t)

        scaler_dn.scale(loss).backward()
        scaler_dn.step(opt_dn)
        scaler_dn.update()

        running += loss.item()
        pbar.set_postfix(loss=f"{running/ (pbar.n or 1):.4f}")
    torch.cuda.empty_cache()

FT_DENOISER = os.path.join(OUT_ROOT, 'mprnet_finetuned.pth')
torch.save(denoiser.state_dict(), FT_DENOISER)


MPRNet | Epoch 1/2:   0%|          | 0/276 [00:00<?, ?it/s]/tmp/ipykernel_19/1491008263.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
MPRNet | Epoch 1/2: 100%|██████████| 276/276 [02:11<00:00,  2.09it/s, loss=0.0074]
MPRNet | Epoch 2/2: 100%|██████████| 276/276 [02:20<00:00,  1.96it/s, loss=0.0070]


In [11]:
denoiser.eval()
for p in denoiser.parameters():
    p.requires_grad_(False)

sr_net.train()
for ep in range(1, SR_EPOCHS+1):
    pbar = tqdm(dl_sr, desc=f"ESRGAN | Epoch {ep}/{SR_EPOCHS}")
    running = 0.0
    for lr_noisy_t, hr_t in pbar:
        lr_noisy_t = lr_noisy_t.to(device, non_blocking=True)
        hr_t       = hr_t.to(device, non_blocking=True)

        with torch.no_grad():
            dn = denoiser(lr_noisy_t)
            if isinstance(dn, (list, tuple)): dn = dn[0]

        opt_sr.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
            sr = sr_net(dn)
            loss = crit_L1(sr, hr_t)

        scaler_sr.scale(loss).backward()
        scaler_sr.step(opt_sr)
        scaler_sr.update()

        running += loss.item()
        pbar.set_postfix(loss=f"{running/ (pbar.n or 1):.4f}")
    torch.cuda.empty_cache()

FT_SR = os.path.join(OUT_ROOT, 'rrdbnet_finetuned.pth')
torch.save(sr_net.state_dict(), FT_SR)


ESRGAN | Epoch 1/2:   0%|          | 0/276 [00:00<?, ?it/s]/tmp/ipykernel_19/3115662338.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
ESRGAN | Epoch 1/2: 100%|██████████| 276/276 [01:47<00:00,  2.56it/s, loss=0.0094]
ESRGAN | Epoch 2/2: 100%|██████████| 276/276 [01:47<00:00,  2.58it/s, loss=0.0095]


In [12]:
def forward_full(dn_model, sr_model, img_np):
    x = to_tensor01(img_np).to(device)
    with torch.no_grad():
        y = dn_model(x)
        if isinstance(y, (list, tuple)): y = y[0]
        z = sr_model(y)
    return to_image_u8(z)

In [13]:
test_paths = sorted([p for p in glob.glob(os.path.join(TEST_DIR, '*')) if os.path.isfile(p)])
for p in tqdm(test_paths, desc="Test inference"):
    img = imread_rgb(p)
    out = forward_full(denoiser, sr_net, img)
    Image.fromarray(out).save(os.path.join(TEST_OUT, os.path.basename(p)))

Test inference: 100%|██████████| 60/60 [00:48<00:00,  1.23it/s]


In [14]:
import os
import numpy as np
import pandas as pd
from PIL import Image

def images_to_csv(folder_path, output_csv):
    data_rows = []
    for filename in tqdm(sorted(os.listdir(folder_path))):
        if filename.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            image_path = os.path.join(folder_path, filename)
            image = Image.open(image_path).convert('L') 
            image_array = np.array(image).flatten()[::8]
            # Replace 'test_' with 'gt_' in the ID
            image_id = filename.split('.')[0].replace('test_', 'gt_')
            data_rows.append([image_id, *image_array])
    column_names = ['ID'] + [f'pixel_{i}' for i in range(len(data_rows[0]) - 1)]
    df = pd.DataFrame(data_rows, columns=column_names)
    df.to_csv(output_csv, index=False)
    print(f'Successfully saved to {output_csv}')

folder_path = '/kaggle/working/outputs/test_out'
output_csv = 'submission.csv'
images_to_csv(folder_path, output_csv)

100%|██████████| 60/60 [00:01<00:00, 50.17it/s]


Successfully saved to submission.csv
